In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

device_torch = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device_torch}")

Using device: cpu


In [2]:
df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')

In [3]:
import pandas as pd
import numpy as np

df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')

print("=== JITTER STATS ===")
print(f"Min: {df['jitter'].min()}")
print(f"Max: {df['jitter'].max()}")
print(f"Mean: {df['jitter'].mean()}")
print(f"Median: {df['jitter'].median()}")
print(f"Std: {df['jitter'].std()}")
print(f"99th percentile: {np.percentile(df['jitter'], 99)}")
print(f"99.9th percentile: {np.percentile(df['jitter'], 99.9)}")
print(f"99.99th percentile: {np.percentile(df['jitter'], 99.99)}")

# Check if jitter was log-transformed
print(f"\n=== IS JITTER LOG-TRANSFORMED? ===")
print(f"Values < 0: {(df['jitter'] < 0).sum()}")
print(f"Values < 1: {(df['jitter'] < 1).sum()}")
print(f"Values > 100: {(df['jitter'] > 100).sum()}")
print(f"Values > 1000: {(df['jitter'] > 1000).sum()}")
print(f"Values > 1000000: {(df['jitter'] > 1000000).sum()}")

# Distribution
print(f"\n=== DISTRIBUTION ===")
print(df['jitter'].describe())

=== JITTER STATS ===
Min: 1.6e-05
Max: 101416259.240383
Mean: 3812.4310799838554
Median: 0.001194
Std: 312329.46730345336
99th percentile: 0.03928930953145026
99.9th percentile: 24879.521966273995
99.99th percentile: 8184409.752654145

=== IS JITTER LOG-TRANSFORMED? ===
Values < 0: 0
Values < 1: 204521
Values > 100: 345
Values > 1000: 277
Values > 1000000: 73

=== DISTRIBUTION ===
count    2.049420e+05
mean     3.812431e+03
std      3.123295e+05
min      1.600000e-05
25%      3.680000e-04
50%      1.194000e-03
75%      3.870000e-03
max      1.014163e+08
Name: jitter, dtype: float64


In [4]:
# Log transform jitter (add small constant to avoid log(0))
jitter_log = np.log1p(df['jitter'])

print("=== JITTER AFTER LOG1P ===")
print(f"Min: {jitter_log.min():.4f}")
print(f"Max: {jitter_log.max():.4f}")
print(f"Mean: {jitter_log.mean():.4f}")
print(f"Median: {jitter_log.median():.4f}")
print(f"Std: {jitter_log.std():.4f}")
print(f"99th percentile: {np.percentile(jitter_log, 99):.4f}")
print(f"99.9th percentile: {np.percentile(jitter_log, 99.9):.4f}")

=== JITTER AFTER LOG1P ===
Min: 0.0000
Max: 18.4347
Mean: 0.0233
Median: 0.0012
Std: 0.4626
99th percentile: 0.0385
99.9th percentile: 10.1207


In [5]:

GAP_THRESHOLD = 60  # seconds

for d in ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']:
    sub = df[df[d] == 1].sort_values('timestamp').reset_index(drop=True)
    gaps = sub['timestamp'].diff().dt.total_seconds()
    break_points = gaps[gaps > GAP_THRESHOLD].index.tolist()
    
    # Segment lengths
    starts = [0] + break_points
    ends = break_points + [len(sub)]
    seg_lengths = [e - s for s, e in zip(starts, ends)]
    
    print(f"\n{d}:")
    print(f"  Total rows: {len(sub)}")
    print(f"  Segments: {len(seg_lengths)}")
    print(f"  Segment lengths: min={min(seg_lengths)}, max={max(seg_lengths)}, "
          f"mean={np.mean(seg_lengths):.0f}, median={np.median(seg_lengths):.0f}")
    print(f"  Segments < 20 rows: {sum(1 for s in seg_lengths if s < 20)}")


device_pc1:
  Total rows: 59519
  Segments: 19
  Segment lengths: min=860, max=4567, mean=3133, median=3154
  Segments < 20 rows: 0

device_pc2:
  Total rows: 42879
  Segments: 14
  Segment lengths: min=545, max=4567, mean=3063, median=3176
  Segments < 20 rows: 0

device_pc3:
  Total rows: 59723
  Segments: 18
  Segment lengths: min=1162, max=4830, mean=3318, median=3234
  Segments < 20 rows: 0

device_pc4:
  Total rows: 42821
  Segments: 13
  Segment lengths: min=1170, max=4828, mean=3294, median=3199
  Segments < 20 rows: 0


In [6]:
df = df.sort_values('timestamp').reset_index(drop=True)

df['sin_COG'] = np.sin(np.radians(df['COG']))
df['cos_COG'] = np.cos(np.radians(df['COG']))

In [7]:
df.shape[0]

204942

In [8]:
df.head(40).where(df['device_pc1'] == 1).dropna()

,timestamp,ping_ms,datarate,jitter,Latitude,Longitude,Altitude,speed_kmh,COG,precipIntensity,...,device_pc2,device_pc3,device_pc4,direction_uplink,measured_qos_delay,hour,day_of_week,date,sin_COG,cos_COG
0,2021-06-22 09:49:54+02:00,7.749322,18.045260,0.000848,52.514013,13.335172,41.9,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
5,2021-06-22 09:49:55+02:00,7.749322,18.201280,0.000152,52.514012,13.335173,41.9,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
10,2021-06-22 09:49:56+02:00,7.749322,18.229520,0.000140,52.514010,13.335173,41.9,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
14,2021-06-22 09:49:57+02:00,7.749322,18.192525,0.000098,52.514007,13.335173,41.9,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
18,2021-06-22 09:49:58+02:00,7.749322,18.376729,0.000438,52.514005,13.335173,41.8,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
22,2021-06-22 09:49:59+02:00,7.255345,18.181154,0.000101,52.514003,13.335173,41.8,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
24,2021-06-22 09:50:00+02:00,7.150202,18.345957,0.000150,52.514002,13.335173,41.8,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
29,2021-06-22 09:50:01+02:00,7.248906,18.336212,0.000168,52.514000,13.335173,41.7,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
33,2021-06-22 09:50:02+02:00,7.271421,18.343800,0.000224,52.513998,13.335173,41.7,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0
36,2021-06-22 09:50:03+02:00,7.261644,18.323068,0.000233,52.513997,13.335172,41.7,0.0,0.0,0.0652,...,0.0,0.0,0.0,0.0,0.0,9.0,1.0,2021-06-22,0.0,1.0


In [9]:
SEQ_LEN = 20
GAP_THRESHOLD = 60
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 256
LR = 0.001
EPOCHS = 50
PATIENCE = 7

In [10]:


BASE_INPUTS = [
    'hour', 'day_of_week',
    'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4',
    'direction_uplink',
    'measured_qos_delay',
    'measurement', 'operator'
]

# Causal chain: now each level includes its own targets as autoregressive inputs
# 'auto_inputs' = targets of THIS level (fed as lagged input from timesteps 1..SEQ_LEN-1)
# 'extra_inputs' = outputs from PREVIOUS levels (fed across all timesteps)
CAUSAL_CHAIN = [
    {
        'name': 'Level_0a_GPS',
        'targets': ['Latitude', 'Longitude'],
        'extra_inputs': [],
        'auto_inputs': ['Latitude', 'Longitude'],
    },
    {
        'name': 'Level_0b_Mobility',
        'targets': ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude'],
        'extra_inputs': ['Latitude', 'Longitude'],
        'auto_inputs': ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude'],
    },
    {
        'name': 'Level_0c_Weather',
        'targets': ['precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed'],
        'extra_inputs': [],
        'auto_inputs': ['precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed'],
    },
    {
        'name': 'Level_0d_Traffic',
        'targets': ['Traffic Jam Factor', 'Traffic Distance'],
        'extra_inputs': ['Latitude', 'Longitude'],
        'auto_inputs': ['Traffic Jam Factor', 'Traffic Distance'],
    },
    {
        'name': 'Level_1_Signal',
        'targets': ['PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                     'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'precipIntensity', 'precipProbability', 'temperature',
                         'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance'],
        'auto_inputs': ['PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                        'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
    },
    {
        'name': 'Level_2_CellConfig',
        'targets': ['PCell_Downlink_frequency', 'PCell_Band_Indicator',
                     'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'precipIntensity', 'precipProbability', 'temperature',
                         'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
        'auto_inputs': ['PCell_Downlink_frequency', 'PCell_Band_Indicator',
                        'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
    },
    {
        'name': 'Level_3a_Downlink',
        'targets': ['PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                     'PCell_Downlink_TB_Size',
                     'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'auto_inputs': ['PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                        'PCell_Downlink_TB_Size',
                        'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High'],
    },
    {
        'name': 'Level_3b_Uplink',
        'targets': ['PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                     'PCell_Uplink_Tx_Power_(dBm)'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'auto_inputs': ['PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                        'PCell_Uplink_Tx_Power_(dBm)'],
    },
    {
        'name': 'Level_4_QoS',
        'targets': ['datarate', 'jitter', 'Pos in Ref Round', 'target_datarate'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                         'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                         'PCell_Downlink_TB_Size',
                         'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
                         'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                         'PCell_Uplink_Tx_Power_(dBm)'],
        'auto_inputs': ['datarate', 'jitter', 'Pos in Ref Round', 'target_datarate'],
    },
    {
        'name': 'Level_5_Ping',
        'targets': ['ping_ms'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                         'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                         'PCell_Downlink_TB_Size',
                         'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
                         'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                         'PCell_Uplink_Tx_Power_(dBm)',
                         'datarate', 'jitter', 'Pos in Ref Round', 'target_datarate'],
        'auto_inputs': ['ping_ms'],
    },
]

SNAP_RULES = {
    'PCell_Downlink_frequency': [125.0, 475.0, 1300.0, 1801.0, 2850.0, 3050.0, 3749.0, 9460.0],
    'PCell_freq_MHz': [700.0, 900.0, 1800.0, 2000.0, 2100.0, 2600.0],
    'PCell_Band_Indicator': [1.0, 3.0, 7.0, 8.0, 28.0],
    'PCell_Downlink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Uplink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Downlink_Average_MCS': list(range(0, 30)),
}



In [11]:
def snap_to_nearest(values, valid_set):
    valid_arr = np.array(valid_set)
    result = np.empty_like(values)
    for i, v in enumerate(values):
        result[i] = valid_arr[np.argmin(np.abs(valid_arr - v))]
    return result

In [12]:
print("\nSegmenting data by device...")

device_cols_list = ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']
all_segments = []

for d_col in device_cols_list:
    sub = df[df[d_col] == 1].sort_values('timestamp').reset_index(drop=True)
    gaps = sub['timestamp'].diff().dt.total_seconds()
    break_indices = gaps[gaps > GAP_THRESHOLD].index.tolist()

    starts = [0] + break_indices
    ends = break_indices + [len(sub)]

    for s, e in zip(starts, ends):
        seg = sub.iloc[s:e].reset_index(drop=True)
        if len(seg) >= SEQ_LEN:
            all_segments.append(seg)

print(f"Total usable segments: {len(all_segments)}")
print(f"Total rows in segments: {sum(len(s) for s in all_segments)}")


Segmenting data by device...
Total usable segments: 64
Total rows in segments: 204942


In [13]:
all_timestamps = df['timestamp'].sort_values()
n = len(all_timestamps)

t_train_end = all_timestamps.iloc[int(n * 0.70)]
t_val_end = all_timestamps.iloc[int(n * 0.85)]

t_train_end = pd.Timestamp(t_train_end)
t_val_end = pd.Timestamp(t_val_end)

print(f"Train cutoff: {t_train_end}")
print(f"Val cutoff:   {t_val_end}")

def get_split(ts):
    ts = pd.Timestamp(ts, tz='Europe/Berlin')
    if ts <= t_train_end:
        return 'train'
    elif ts <= t_val_end:
        return 'val'
    else:
        return 'test'

Train cutoff: 2021-06-23 15:51:37+02:00
Val cutoff:   2021-06-24 10:19:02+02:00


In [14]:
def create_sequences_autoreg(segments, context_cols, auto_cols, target_cols, split='train'):
    """
    Create sequences where:
    - context_cols: base + extra inputs (same across all timesteps, from ground truth or previous levels)
    - auto_cols: autoregressive inputs = the targets themselves from previous timesteps
    - target_cols: what to predict at the LAST timestep
    
    Input at each timestep = context_cols + auto_cols
    But for the LAST timestep, auto_cols are NOT included (that's what we predict)
    So: timesteps 0..SEQ_LEN-2 have full input, timestep SEQ_LEN-1 has only context_cols
    
    To keep dimensions consistent, we MASK auto_cols at the last timestep to 0.
    """
    X_list, y_list, ts_list = [], [], []
    
    all_input_cols = context_cols + auto_cols
    
    for seg_df in segments:
        seg_timestamps = seg_df['timestamp'].values
        seg_context = seg_df[context_cols].values
        seg_auto = seg_df[auto_cols].values
        seg_targets = seg_df[target_cols].values
        
        for i in range(SEQ_LEN, len(seg_df)):
            target_ts = seg_timestamps[i]
            if get_split(pd.Timestamp(target_ts)) != split:
                continue
            
            # Window: rows [i-SEQ_LEN, i-SEQ_LEN+1, ..., i-1] for input
            # Target: row i
            
            # Context for all SEQ_LEN timesteps (rows i-SEQ_LEN to i-1)
            ctx = seg_context[i - SEQ_LEN:i]  # (SEQ_LEN, n_context)
            
            # Autoregressive: use rows i-SEQ_LEN to i-1
            # These ARE the target values from previous timesteps
            auto = seg_auto[i - SEQ_LEN:i].copy()  # (SEQ_LEN, n_auto)
            
            # Mask last timestep's autoregressive values to 0
            # (we don't know these yet — they're what we're predicting)
            auto[-1, :] = 0.0
            
            # Combine
            x = np.concatenate([ctx, auto], axis=1)  # (SEQ_LEN, n_context + n_auto)
            y = seg_targets[i]  # (n_targets,)
            
            X_list.append(x)
            y_list.append(y)
            ts_list.append(target_ts)
    
    if len(X_list) == 0:
        return None, None, None
    
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y, ts_list

In [15]:
class CausalGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )

    def forward(self, x):
        gru_out, _ = self.gru(x)
        last_hidden = gru_out[:, -1, :]
        return self.fc(last_hidden)

In [16]:
def train_level(level_name, context_cols, auto_cols, target_cols, segments, scalers_dict):
    print(f"\n{'─'*60}")
    print(f"  {level_name}")
    print(f"  Targets: {target_cols}")
    print(f"  Context features: {len(context_cols)}, Autoregressive: {len(auto_cols)}")
    print(f"  Total input per timestep: {len(context_cols) + len(auto_cols)}")
    print(f"{'─'*60}")

    t0 = time.time()
    
    X_train, y_train, _ = create_sequences_autoreg(segments, context_cols, auto_cols, target_cols, 'train')
    X_val, y_val, _ = create_sequences_autoreg(segments, context_cols, auto_cols, target_cols, 'val')
    X_test, y_test, ts_test = create_sequences_autoreg(segments, context_cols, auto_cols, target_cols, 'test')

    print(f"  Sequences — Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

    # Scale inputs
    n_feat = X_train.shape[2]
    scaler = StandardScaler()
    X_train_2d = X_train.reshape(-1, n_feat)
    X_train_2d = scaler.fit_transform(X_train_2d)
    X_train = X_train_2d.reshape(-1, SEQ_LEN, n_feat)

    X_val = scaler.transform(X_val.reshape(-1, n_feat)).reshape(-1, SEQ_LEN, n_feat)
    X_test = scaler.transform(X_test.reshape(-1, n_feat)).reshape(-1, SEQ_LEN, n_feat)

    scalers_dict[f"{level_name}_input"] = scaler

    # Scale targets
    target_scaler = StandardScaler()
    y_train_scaled = target_scaler.fit_transform(y_train)
    y_val_scaled = target_scaler.transform(y_val)
    scalers_dict[f"{level_name}_target"] = target_scaler

    # DataLoaders
    train_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_train), torch.FloatTensor(y_train_scaled))
    val_dataset = torch.utils.data.TensorDataset(
        torch.FloatTensor(X_val), torch.FloatTensor(y_val_scaled))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Model
    model = CausalGRU(
        input_dim=n_feat,
        hidden_dim=HIDDEN_SIZE,
        output_dim=len(target_cols),
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(device_torch)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device_torch), y_batch.to(device_torch)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(X_batch)
        train_loss /= len(train_dataset)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device_torch), y_batch.to(device_torch)
                pred = model(X_batch)
                loss = criterion(pred, y_batch)
                val_loss += loss.item() * len(X_batch)
        val_loss /= len(val_dataset)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or patience_counter == 0:
            print(f"  Epoch {epoch+1:3d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | Pat: {patience_counter}")

        if patience_counter >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    model.eval()

    # Evaluate on test (teacher-forced)
    X_test_tensor = torch.FloatTensor(X_test).to(device_torch)
    with torch.no_grad():
        pred_test_scaled = model(X_test_tensor).cpu().numpy()
    pred_test = target_scaler.inverse_transform(pred_test_scaled)

    elapsed = time.time() - t0

    level_results = {}
    for i, target in enumerate(target_cols):
        pred_col = pred_test[:, i]
        if target in SNAP_RULES:
            pred_col = snap_to_nearest(pred_col, SNAP_RULES[target])

        rmse = np.sqrt(mean_squared_error(y_test[:, i], pred_col))
        mae = mean_absolute_error(y_test[:, i], pred_col)

        level_results[target] = {
            'level': level_name,
            'rmse_test_tf': rmse,
            'mae_test_tf': mae,
        }
        print(f"  {target:40s} | Test RMSE(TF): {rmse:10.4f} | MAE: {mae:10.4f}")

    print(f"  Training time: {elapsed:.1f}s")
    return model, level_results

In [17]:
print("TRAINING GRU CAUSAL CHAIN (Autoregressive + Teacher-Forced)")
print("="*80)

all_models = {}
all_results = {}
all_scalers = {}
total_start = time.time()

for level in CAUSAL_CHAIN:
    level_name = level['name']
    targets = level['targets']
    extra_inputs = level['extra_inputs']
    auto_inputs = level['auto_inputs']
    
    context_cols = BASE_INPUTS + extra_inputs

    model, level_results = train_level(
        level_name, context_cols, auto_inputs, targets, all_segments, all_scalers
    )

    all_models[level_name] = model
    all_results.update(level_results)

total_time = time.time() - total_start

TRAINING GRU CAUSAL CHAIN (Autoregressive + Teacher-Forced)

────────────────────────────────────────────────────────────
  Level_0a_GPS
  Targets: ['Latitude', 'Longitude']
  Context features: 10, Autoregressive: 2
  Total input per timestep: 12
────────────────────────────────────────────────────────────
  Sequences — Train: 157599, Val: 26910, Test: 19153
  Epoch   1 | Train: 0.784239 | Val: 2.025884 | LR: 0.001000 | Pat: 0
  Epoch   5 | Train: 0.292203 | Val: 2.905001 | LR: 0.000500 | Pat: 4
  Early stop at epoch 8
  Latitude                                 | Test RMSE(TF):     0.0116 | MAE:     0.0089
  Longitude                                | Test RMSE(TF):     0.0476 | MAE:     0.0410
  Training time: 966.2s

────────────────────────────────────────────────────────────
  Level_0b_Mobility
  Targets: ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude']
  Context features: 12, Autoregressive: 4
  Total input per timestep: 16
───────────────────────────────────────────────────────────

In [18]:
if 'sin_COG' in all_results and 'cos_COG' in all_results:
    print(f"\nCOG: sin RMSE={all_results['sin_COG']['rmse_test_tf']:.4f}, "
          f"cos RMSE={all_results['cos_COG']['rmse_test_tf']:.4f}")


COG: sin RMSE=0.0787, cos RMSE=0.0712


In [19]:
print(f"\n{'='*80}")
print(f"GRU CAUSAL CHAIN v2 — FULL RESULTS (Teacher-Forced + Autoregressive)")
print(f"{'='*80}")
print(f"{'Feature':45s} | {'Level':20s} | {'Test RMSE':>12s} | {'Test MAE':>12s}")
print("─" * 95)
for feat, r in sorted(all_results.items(), key=lambda x: x[1]['level']):
    print(f"{feat:45s} | {r['level']:20s} | {r['rmse_test_tf']:12.4f} | {r['mae_test_tf']:12.4f}")

print(f"\nTOTAL TRAINING TIME: {total_time:.1f}s ({total_time/60:.1f} min)")


GRU CAUSAL CHAIN v2 — FULL RESULTS (Teacher-Forced + Autoregressive)
Feature                                       | Level                |    Test RMSE |     Test MAE
───────────────────────────────────────────────────────────────────────────────────────────────
Latitude                                      | Level_0a_GPS         |       0.0116 |       0.0089
Longitude                                     | Level_0a_GPS         |       0.0476 |       0.0410
speed_kmh                                     | Level_0b_Mobility    |       0.4538 |       0.2781
sin_COG                                       | Level_0b_Mobility    |       0.0787 |       0.0417
cos_COG                                       | Level_0b_Mobility    |       0.0712 |       0.0382
Altitude                                      | Level_0b_Mobility    |       1.3647 |       1.2806
precipIntensity                               | Level_0c_Weather     |       0.1138 |       0.0909
precipProbability                         

In [20]:

torch.save({name: m.state_dict() for name, m in all_models.items()}, 'gru_causal_v2_models.pt')
pickle.dump(all_results, open('gru_causal_v2_results.pkl', 'wb'))
pickle.dump(all_scalers, open('gru_causal_v2_scalers.pkl', 'wb'))

print("\nSaved: gru_causal_v2_models.pt, gru_causal_v2_results.pkl, gru_causal_v2_scalers.pkl")
print("Done!")


Saved: gru_causal_v2_models.pt, gru_causal_v2_results.pkl, gru_causal_v2_scalers.pkl
Done!


In [21]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

device_torch = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device_torch}")

Using device: cpu


In [23]:

df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')
df = df.sort_values('timestamp').reset_index(drop=True)
df['sin_COG'] = np.sin(np.radians(df['COG']))
df['cos_COG'] = np.cos(np.radians(df['COG']))
df['jitter_log'] = np.log1p(df['jitter'])

In [27]:
SEQ_LEN = 20
GAP_THRESHOLD = 60
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 256
LR = 0.001
EPOCHS = 50
PATIENCE = 7

BASE_INPUTS = [
    'hour', 'day_of_week',
    'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4',
    'direction_uplink',
    'measured_qos_delay',
    'measurement', 'operator'
]

SNAP_RULES = {
    'PCell_Downlink_frequency': [125.0, 475.0, 1300.0, 1801.0, 2850.0, 3050.0, 3749.0, 9460.0],
    'PCell_freq_MHz': [700.0, 900.0, 1800.0, 2000.0, 2100.0, 2600.0],
    'PCell_Band_Indicator': [1.0, 3.0, 7.0, 8.0, 28.0],
    'PCell_Downlink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Uplink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Downlink_Average_MCS': list(range(0, 30)),
}

def snap_to_nearest(values, valid_set):
    valid_arr = np.array(valid_set)
    result = np.empty_like(values)
    for i, v in enumerate(values):
        result[i] = valid_arr[np.argmin(np.abs(valid_arr - v))]
    return result

# Full causal chain definition
CAUSAL_CHAIN = [
    {
        'name': 'Level_0a_GPS',
        'targets': ['Latitude', 'Longitude'],
        'extra_inputs': [],
        'auto_inputs': ['Latitude', 'Longitude'],
    },
    {
        'name': 'Level_0b_Mobility',
        'targets': ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude'],
        'extra_inputs': ['Latitude', 'Longitude'],
        'auto_inputs': ['speed_kmh', 'sin_COG', 'cos_COG', 'Altitude'],
    },
    {
        'name': 'Level_0c_Weather',
        'targets': ['precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed'],
        'extra_inputs': [],
        'auto_inputs': ['precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed'],
    },
    {
        'name': 'Level_0d_Traffic',
        'targets': ['Traffic Jam Factor', 'Traffic Distance'],
        'extra_inputs': ['Latitude', 'Longitude'],
        'auto_inputs': ['Traffic Jam Factor', 'Traffic Distance'],
    },
    {
        'name': 'Level_1_Signal',
        'targets': ['PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                     'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'precipIntensity', 'precipProbability', 'temperature',
                         'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance'],
        'auto_inputs': ['PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                        'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
    },
    {
        'name': 'Level_2_CellConfig',
        'targets': ['PCell_Downlink_frequency', 'PCell_Band_Indicator',
                     'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'precipIntensity', 'precipProbability', 'temperature',
                         'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz'],
        'auto_inputs': ['PCell_Downlink_frequency', 'PCell_Band_Indicator',
                        'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
    },
    {
        'name': 'Level_3a_Downlink',
        'targets': ['PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                     'PCell_Downlink_TB_Size',
                     'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'auto_inputs': ['PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                        'PCell_Downlink_TB_Size',
                        'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High'],
    },
    {
        'name': 'Level_3b_Uplink',
        'targets': ['PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                     'PCell_Uplink_Tx_Power_(dBm)'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz'],
        'auto_inputs': ['PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                        'PCell_Uplink_Tx_Power_(dBm)'],
    },
    {
        'name': 'Level_4_QoS',
        'targets': ['datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                         'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                         'PCell_Downlink_TB_Size',
                         'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
                         'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                         'PCell_Uplink_Tx_Power_(dBm)'],
        'auto_inputs': ['datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate'],
    },
    {
        'name': 'Level_5_Ping',
        'targets': ['ping_ms'],
        'extra_inputs': ['Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG',
                         'Altitude', 'Traffic Jam Factor', 'Traffic Distance',
                         'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
                         'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
                         'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                         'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                         'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs',
                         'PCell_Downlink_TB_Size',
                         'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
                         'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size',
                         'PCell_Uplink_Tx_Power_(dBm)',
                         'datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate'],
        'auto_inputs': ['ping_ms'],
    },
]


In [28]:
device_cols_list = ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']
all_segments = []
for d_col in device_cols_list:
    sub = df[df[d_col] == 1].sort_values('timestamp').reset_index(drop=True)
    gaps = sub['timestamp'].diff().dt.total_seconds()
    break_indices = gaps[gaps > GAP_THRESHOLD].index.tolist()
    starts = [0] + break_indices
    ends = break_indices + [len(sub)]
    for s, e in zip(starts, ends):
        seg = sub.iloc[s:e].reset_index(drop=True)
        if len(seg) >= SEQ_LEN:
            all_segments.append(seg)

print(f"Segments: {len(all_segments)}, Total rows: {sum(len(s) for s in all_segments)}")

all_timestamps = df['timestamp'].sort_values()
n = len(all_timestamps)
t_train_end = all_timestamps.iloc[int(n * 0.70)]
t_val_end = all_timestamps.iloc[int(n * 0.85)]

def get_split(ts):
    ts = pd.Timestamp(ts, tz='Europe/Berlin')
    if ts <= t_train_end:
        return 'train'
    elif ts <= t_val_end:
        return 'val'
    else:
        return 'test'

Segments: 64, Total rows: 204942


In [29]:


class CausalGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim,
                          num_layers=num_layers,
                          dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    def forward(self, x):
        gru_out, _ = self.gru(x)
        return self.fc(gru_out[:, -1, :])

saved_states = torch.load('gru_causal_v2_models.pt', map_location=device_torch)
saved_scalers = pickle.load(open('gru_causal_v2_scalers.pkl', 'rb'))

# Rebuild models for levels 0-3b (load saved weights)
all_models = {}
all_scalers = {}

levels_to_load = ['Level_0a_GPS', 'Level_0b_Mobility', 'Level_0c_Weather',
                  'Level_0d_Traffic', 'Level_1_Signal', 'Level_2_CellConfig',
                  'Level_3a_Downlink', 'Level_3b_Uplink']

for level in CAUSAL_CHAIN:
    if level['name'] not in levels_to_load:
        continue

    level_name = level['name']
    context_cols = BASE_INPUTS + level['extra_inputs']
    auto_cols = level['auto_inputs']
    n_input = len(context_cols) + len(auto_cols)
    n_output = len(level['targets'])

    model = CausalGRU(n_input, HIDDEN_SIZE, n_output, NUM_LAYERS, DROPOUT).to(device_torch)
    model.load_state_dict(saved_states[level_name])
    model.eval()
    all_models[level_name] = model

    # Load scalers
    all_scalers[f"{level_name}_input"] = saved_scalers[f"{level_name}_input"]
    all_scalers[f"{level_name}_target"] = saved_scalers[f"{level_name}_target"]

    print(f"  Loaded {level_name} (input={n_input}, output={n_output})")

# Store TF results from previous run for comparison
prev_tf_results = pickle.load(open('gru_causal_v2_results.pkl', 'rb'))

  Loaded Level_0a_GPS (input=12, output=2)
  Loaded Level_0b_Mobility (input=16, output=4)
  Loaded Level_0c_Weather (input=15, output=5)
  Loaded Level_0d_Traffic (input=14, output=2)
  Loaded Level_1_Signal (input=29, output=6)
  Loaded Level_2_CellConfig (input=33, output=4)
  Loaded Level_3a_Downlink (input=34, output=6)
  Loaded Level_3b_Uplink (input=31, output=3)


In [30]:
def create_sequences_autoreg(segments, context_cols, auto_cols, target_cols, split='train'):
    X_list, y_list = [], []
    for seg_df in segments:
        seg_timestamps = seg_df['timestamp'].values
        seg_context = seg_df[context_cols].values
        seg_auto = seg_df[auto_cols].values
        seg_targets = seg_df[target_cols].values
        for i in range(SEQ_LEN, len(seg_df)):
            if get_split(pd.Timestamp(seg_timestamps[i])) != split:
                continue
            ctx = seg_context[i - SEQ_LEN:i]
            auto = seg_auto[i - SEQ_LEN:i].copy()
            auto[-1, :] = 0.0
            x = np.concatenate([ctx, auto], axis=1)
            y = seg_targets[i]
            X_list.append(x)
            y_list.append(y)
    if not X_list:
        return None, None
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

In [31]:
def train_level_fresh(level_name, context_cols, auto_cols, target_cols, segments, scalers_dict):
    print(f"\n{'─'*60}")
    print(f"  RETRAINING: {level_name}")
    print(f"  Targets: {target_cols}")
    print(f"{'─'*60}")

    t0 = time.time()
    X_train, y_train = create_sequences_autoreg(segments, context_cols, auto_cols, target_cols, 'train')
    X_val, y_val = create_sequences_autoreg(segments, context_cols, auto_cols, target_cols, 'val')
    X_test, y_test = create_sequences_autoreg(segments, context_cols, auto_cols, target_cols, 'test')
    print(f"  Sequences — Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

    n_feat = X_train.shape[2]
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train.reshape(-1, n_feat)).reshape(-1, SEQ_LEN, n_feat)
    X_val = scaler.transform(X_val.reshape(-1, n_feat)).reshape(-1, SEQ_LEN, n_feat)
    X_test = scaler.transform(X_test.reshape(-1, n_feat)).reshape(-1, SEQ_LEN, n_feat)
    scalers_dict[f"{level_name}_input"] = scaler

    target_scaler = StandardScaler()
    y_train_s = target_scaler.fit_transform(y_train)
    y_val_s = target_scaler.transform(y_val)
    scalers_dict[f"{level_name}_target"] = target_scaler

    train_loader = DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train_s)),
        batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(
        torch.utils.data.TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val_s)),
        batch_size=BATCH_SIZE, shuffle=False)

    model = CausalGRU(n_feat, HIDDEN_SIZE, len(target_cols), NUM_LAYERS, DROPOUT).to(device_torch)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    pat = 0

    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device_torch), yb.to(device_torch)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item() * len(xb)
        t_loss /= len(X_train)

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device_torch), yb.to(device_torch)
                v_loss += criterion(model(xb), yb).item() * len(xb)
        v_loss /= len(X_val)
        scheduler.step(v_loss)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1

        if (epoch + 1) % 5 == 0 or pat == 0:
            print(f"  Epoch {epoch+1:3d} | Train: {t_loss:.6f} | Val: {v_loss:.6f} | Pat: {pat}")

        if pat >= PATIENCE:
            print(f"  Early stop at epoch {epoch+1}")
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_s = model(torch.FloatTensor(X_test).to(device_torch)).cpu().numpy()
    pred = target_scaler.inverse_transform(pred_s)

    elapsed = time.time() - t0
    tf_results = {}
    for i, t in enumerate(target_cols):
        p = pred[:, i].copy()
        if t in SNAP_RULES:
            p = snap_to_nearest(p, SNAP_RULES[t])
        rmse = np.sqrt(mean_squared_error(y_test[:, i], p))
        mae = mean_absolute_error(y_test[:, i], p)
        tf_results[t] = {'rmse_tf': rmse, 'mae_tf': mae}
        print(f"  {t:40s} | RMSE(TF): {rmse:10.4f} | MAE: {mae:10.4f}")
    print(f"  Time: {elapsed:.1f}s")

    return model, tf_results



# Retrain Level 4
level4 = CAUSAL_CHAIN[8]  # Level_4_QoS
ctx4 = BASE_INPUTS + level4['extra_inputs']
model4, tf4 = train_level_fresh(level4['name'], ctx4, level4['auto_inputs'],
                                 level4['targets'], all_segments, all_scalers)
all_models[level4['name']] = model4

# Retrain Level 5
level5 = CAUSAL_CHAIN[9]  # Level_5_Ping
ctx5 = BASE_INPUTS + level5['extra_inputs']
model5, tf5 = train_level_fresh(level5['name'], ctx5, level5['auto_inputs'],
                                 level5['targets'], all_segments, all_scalers)
all_models[level5['name']] = model5

# Merge TF results
all_tf_results = {}
for col, r in prev_tf_results.items():
    if col != 'jitter':  # skip old jitter
        all_tf_results[col] = {'rmse_tf': r['rmse_test_tf'], 'mae_tf': r['mae_test_tf']}
all_tf_results.update(tf4)
all_tf_results.update(tf5)


────────────────────────────────────────────────────────────
  RETRAINING: Level_4_QoS
  Targets: ['datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate']
────────────────────────────────────────────────────────────
  Sequences — Train: 157599, Val: 26910, Test: 19153
  Epoch   1 | Train: 0.254568 | Val: 0.063846 | Pat: 0
  Epoch   2 | Train: 0.208304 | Val: 0.056067 | Pat: 0
  Epoch   5 | Train: 0.203539 | Val: 0.061429 | Pat: 3
  Epoch   7 | Train: 0.196629 | Val: 0.055436 | Pat: 0
  Epoch  10 | Train: 0.142669 | Val: 0.071271 | Pat: 3
  Epoch  11 | Train: 0.131137 | Val: 0.053667 | Pat: 0
  Epoch  15 | Train: 0.103641 | Val: 0.059383 | Pat: 4
  Early stop at epoch 18
  datarate                                 | RMSE(TF):     0.5057 | MAE:     0.3838
  jitter_log                               | RMSE(TF):     0.5654 | MAE:     0.0605
  Pos in Ref Round                         | RMSE(TF):     0.1730 | MAE:     0.0683
  target_datarate                          | RMSE(TF):     

In [33]:

cascade_start = time.time()

all_generated_cols = []
for level in CAUSAL_CHAIN:
    all_generated_cols.extend(level['targets'])

all_true = {col: [] for col in all_generated_cols}
all_pred = {col: [] for col in all_generated_cols}

test_seg_count = 0
test_row_count = 0

for seg_idx, seg_df in enumerate(all_segments):
    seg_timestamps = seg_df['timestamp'].values
    test_mask = np.array([get_split(pd.Timestamp(t)) == 'test' for t in seg_timestamps])
    if test_mask.sum() == 0:
        continue

    test_seg_count += 1

    # Initialize gen_buffer with ground truth
    gen_buffer = {}
    for col in all_generated_cols:
        gen_buffer[col] = seg_df[col].values.copy().astype(np.float64)

    for i in range(SEQ_LEN, len(seg_df)):
        if not test_mask[i]:
            continue

        test_row_count += 1

        for level in CAUSAL_CHAIN:
            level_name = level['name']
            context_cols = BASE_INPUTS + level['extra_inputs']
            auto_cols = level['auto_inputs']
            target_cols = level['targets']

            model = all_models[level_name]
            input_scaler = all_scalers[f"{level_name}_input"]
            target_scaler = all_scalers[f"{level_name}_target"]

            # Build context sequence
            ctx_seq = np.zeros((SEQ_LEN, len(context_cols)), dtype=np.float32)
            for t in range(SEQ_LEN):
                row_idx = i - SEQ_LEN + t
                for j, col in enumerate(context_cols):
                    if col in BASE_INPUTS:
                        ctx_seq[t, j] = seg_df[col].iloc[row_idx]
                    else:
                        ctx_seq[t, j] = gen_buffer[col][row_idx]

            # Build autoregressive sequence
            auto_seq = np.zeros((SEQ_LEN, len(auto_cols)), dtype=np.float32)
            for t in range(SEQ_LEN):
                row_idx = i - SEQ_LEN + t
                for j, col in enumerate(auto_cols):
                    auto_seq[t, j] = gen_buffer[col][row_idx]
            auto_seq[-1, :] = 0.0

            # Combine, scale, predict
            x = np.concatenate([ctx_seq, auto_seq], axis=1)
            x_scaled = input_scaler.transform(x.reshape(-1, x.shape[1])).reshape(1, SEQ_LEN, -1)

            with torch.no_grad():
                pred_scaled = model(torch.FloatTensor(x_scaled).to(device_torch)).cpu().numpy()
            pred = target_scaler.inverse_transform(pred_scaled)[0]

            # Snap and store
            for j, col in enumerate(target_cols):
                val = pred[j]
                if col in SNAP_RULES:
                    val = snap_to_nearest(np.array([val]), SNAP_RULES[col])[0]
                gen_buffer[col][i] = val

        # Record
        for col in all_generated_cols:
            all_true[col].append(seg_df[col].iloc[i])
            all_pred[col].append(gen_buffer[col][i])

    if test_seg_count % 5 == 0:
        print(f"  Processed {test_seg_count} segments, {test_row_count} rows...")

cascade_time = time.time() - cascade_start
print(f"\nCascaded eval: {test_seg_count} segments, {test_row_count} rows in {cascade_time:.1f}s")

  Processed 5 segments, 13421 rows...

Cascaded eval: 7 segments, 19153 rows in 1871.9s


In [34]:
print(f"\n{'='*80}")
print(f"GRU v2 FINAL: Teacher-Forced vs Cascaded")
print(f"{'='*80}")
print(f"{'Feature':45s} | {'RMSE(TF)':>10s} | {'RMSE(Casc)':>11s} | {'MAE(Casc)':>10s} | {'Ratio':>6s}")
print("─" * 100)

for col in all_generated_cols:
    true_arr = np.array(all_true[col])
    pred_arr = np.array(all_pred[col])

    # For jitter_log, also show raw scale
    if col == 'jitter_log':
        # Log space metrics
        rmse_casc = np.sqrt(mean_squared_error(true_arr, pred_arr))
        mae_casc = mean_absolute_error(true_arr, pred_arr)
        rmse_tf = all_tf_results[col]['rmse_tf'] if col in all_tf_results else 0
        ratio = rmse_casc / rmse_tf if rmse_tf > 0 else 0
        print(f"{'jitter_log (log space)':45s} | {rmse_tf:10.4f} | {rmse_casc:11.4f} | {mae_casc:10.4f} | {ratio:5.2f}x")

        # Raw scale
        true_raw = np.expm1(true_arr)
        pred_raw = np.expm1(pred_arr)
        rmse_raw = np.sqrt(mean_squared_error(true_raw, pred_raw))
        mae_raw = mean_absolute_error(true_raw, pred_raw)
        print(f"{'jitter (raw from log)':45s} | {'n/a':>10s} | {rmse_raw:11.4f} | {mae_raw:10.4f} | {'':>6s}")
    else:
        rmse_casc = np.sqrt(mean_squared_error(true_arr, pred_arr))
        mae_casc = mean_absolute_error(true_arr, pred_arr)
        rmse_tf = all_tf_results.get(col, {}).get('rmse_tf', 0)
        ratio = rmse_casc / rmse_tf if rmse_tf > 0 else 0
        print(f"{col:45s} | {rmse_tf:10.4f} | {rmse_casc:11.4f} | {mae_casc:10.4f} | {ratio:5.2f}x")

# COG recovery
if 'sin_COG' in all_pred and 'cos_COG' in all_pred:
    pred_cog = np.degrees(np.arctan2(np.array(all_pred['sin_COG']),
                                      np.array(all_pred['cos_COG']))) % 360
    true_cog_vals = np.array(all_true['sin_COG'])
    # Get actual COG from segments for test rows — use atan2 on true sin/cos
    true_cog = np.degrees(np.arctan2(np.array(all_true['sin_COG']),
                                      np.array(all_true['cos_COG']))) % 360
    diff = np.abs(true_cog - pred_cog)
    circular_diff = np.minimum(diff, 360 - diff)
    print(f"\n{'COG (recovered, circular)':45s} | {'':>10s} | {np.sqrt(np.mean(circular_diff**2)):11.4f} | {np.mean(circular_diff):10.4f} |")

print(f"\nCascade eval time: {cascade_time:.1f}s ({cascade_time/60:.1f} min)")


GRU v2 FINAL: Teacher-Forced vs Cascaded
Feature                                       |   RMSE(TF) |  RMSE(Casc) |  MAE(Casc) |  Ratio
────────────────────────────────────────────────────────────────────────────────────────────────────
Latitude                                      |     0.0116 |      0.0116 |     0.0089 |  1.00x
Longitude                                     |     0.0476 |      0.0476 |     0.0410 |  1.00x
speed_kmh                                     |     0.4538 |      2.2037 |     1.8830 |  4.86x
sin_COG                                       |     0.0787 |      1.0559 |     0.7684 | 13.42x
cos_COG                                       |     0.0712 |      0.6228 |     0.5063 |  8.74x
Altitude                                      |     1.3647 |     22.1690 |    20.7844 | 16.24x
precipIntensity                               |     0.1138 |      0.0981 |     0.0814 |  0.86x
precipProbability                             |     0.2175 |      0.2808 |     0.2607 |  1.29x
te

In [36]:


torch.save({name: m.state_dict() for name, m in all_models.items()}, 'research\generation_output/gru_v2_final_models.pt')
pickle.dump(all_tf_results, open('research\generation_output/gru_v2_final_tf_results.pkl', 'wb'))
pickle.dump(all_scalers, open('research\generation_output/gru_v2_final_scalers.pkl', 'wb'))
pickle.dump({'true': all_true, 'pred': all_pred}, open('research\generation_output/gru_v2_cascade_predictions.pkl', 'wb'))

print("\nSaved all models, results, scalers, and predictions.")
print("Done!")


Saved all models, results, scalers, and predictions.
Done!
